# 01 · Clean the trips and build the tables

**Who Gets a Bluebike?** · CP 101 final project · George Huang, Oliver Kirk, Handson Li, Shaan Patel

This notebook turns the raw 2024 Bluebikes trips and the census data into small, clean tables.

**Run `00_download_trips.ipynb` first.**

| | Files |
|---|---|
| In | `data/raw/bluebikes/` (12 monthly trip CSVs), `data/raw/acs/`, `data/raw/boundaries/` |
| Out | `data/clean/` (small tables, on GitHub) and `data/interim/trips/` (clean trips, one file per month, too big for GitHub) |

It works **one month at a time**. Each month is cleaned, summarized and saved before the next one is read, so the computer never holds all 4.7 million trips at once.

## Setup

In [1]:
from pathlib import Path
import json

import pandas as pd
import geopandas as gpd

RAW_TRIPS = Path("../data/raw/bluebikes")
RAW_ACS = Path("../data/raw/acs/acs5_suffolk_tracts.json")
RAW_BOUNDARIES = Path("../data/raw/boundaries")
INTERIM_TRIPS = Path("../data/interim/trips")
CLEAN = Path("../data/clean")
INTERIM_TRIPS.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)

STUDY_YEAR = 2024
MIN_MINUTES = 1            # shorter trips are removed; see rule 2 below
MAX_MINUTES = 24 * 60      # longer trips are removed; see rule 3 below
LOCAL_CRS = 2249           # NAD83 / Massachusetts Mainland, in US survey feet (Boston's version of EPSG:2227)
RESIDENTIAL_MIN_POP = 500  # tracts with fewer people are parks, the airport or islands, not neighborhoods

In [2]:
trip_files = sorted(RAW_TRIPS.glob(str(STUDY_YEAR) + "*-bluebikes-tripdata.csv"))

print(len(trip_files), "monthly files")
assert len(trip_files) == 12, "Expected 12 monthly files. Run 00_download_trips.ipynb first." 

12 monthly files


## 1. Clean and summarize each month

For each monthly file, the loop below

1. reads the file and turns the start and end times into datetimes;
2. removes rows by five rules, counting how many each rule removes;
3. saves the clean trips;
4. makes three small summary tables for that month, which are combined after the loop.

| Rule | Why the rows are removed |
|---|---|
| 1. No start station | We cannot place the trip on the map. |
| 2. Shorter than 1 minute | January to May include these trips and June to December do not, because Bluebikes changed how it exports data in June 2024. Removing them from every month keeps the months comparable. This also removes the few trips that seem to end before they start. |
| 3. Longer than 24 hours | Almost always a lost or unreturned bike, not a ride. |
| 4. Started outside 2024 | A file can include trips that started just before midnight at the end of the previous month. |
| 5. Already in the previous month's file | 123 rides that crossed midnight on May 31 appear in **both** the May and the June file. They are the same trips, so we keep the first copy, as in Lab 6. A ride lasts at most 24 hours, so a copy can only ever be in the neighboring file. |

Trips with a start station but no end station (about 0.1%) are **kept**, because we count trips by where they start.

In [3]:
cleaning_log = []       # one dictionary of counts per file
monthly_starts = []     # small summary tables, one per month, combined after the loop
monthly_ends = []
monthly_stations = []
previous_month = None   # the clean trips from the month before, for rule 5

for path in trip_files:
    trips = pd.read_csv(path, dtype={"start_station_id": str, "end_station_id": str})
    log = {"file": path.name, "rows_in": len(trips)}

    # text timestamps to datetimes, then each trip's length in minutes and its month
    trips["started_at"] = pd.to_datetime(trips["started_at"])
    trips["ended_at"] = pd.to_datetime(trips["ended_at"])
    trips["duration_min"] = (trips["ended_at"] - trips["started_at"]).dt.total_seconds() / 60
    trips["month"] = trips["started_at"].dt.strftime("%Y-%m")

    # rule 1: no start station
    rows_before = len(trips)
    trips = trips[trips["start_station_id"].notna()]
    log["no_start_station"] = rows_before - len(trips)

    # rule 2: shorter than 1 minute
    rows_before = len(trips)
    trips = trips[trips["duration_min"] >= MIN_MINUTES]
    log["under_1_min"] = rows_before - len(trips)

    # rule 3: longer than 24 hours
    rows_before = len(trips)
    trips = trips[trips["duration_min"] <= MAX_MINUTES]
    log["over_24_hours"] = rows_before - len(trips)

    # rule 4: started outside the study year
    rows_before = len(trips)
    trips = trips[trips["started_at"].dt.year == STUDY_YEAR]
    log["outside_study_year"] = rows_before - len(trips)

    # rule 5: already in the previous month's file
    rows_before = len(trips)
    if previous_month is not None:
        trips = trips[~trips["ride_id"].isin(previous_month["ride_id"])]
    log["in_previous_file"] = rows_before - len(trips)
    previous_month = trips

    log["rows_out"] = len(trips)
    cleaning_log.append(log)
    print(path.name, "kept", len(trips), "of", log["rows_in"])

    # True/False columns, so that summing them counts trips
    trips["is_member"] = trips["member_casual"] == "member"
    trips["is_ebike"] = trips["rideable_type"] == "electric_bike"
    trips["is_weekend"] = trips["started_at"].dt.dayofweek >= 5   # Monday is 0, so 5 and 6 are Saturday and Sunday
    trips["hour"] = trips["started_at"].dt.hour

    # save this month's clean trips for later analysis
    trips.to_parquet(INTERIM_TRIPS / (path.stem + ".parquet"), index=False)

    # trips started at each station this month
    starts = trips.groupby(["start_station_id", "month"]).agg(
        trips_started=("ride_id", "size"),
        member_trips=("is_member", "sum"),
        ebike_trips=("is_ebike", "sum"),
        weekend_trips=("is_weekend", "sum"),
    ).reset_index()
    monthly_starts.append(starts)

    # trips ended at each station this month (trips with no end station are left out)
    ends = trips.groupby(["end_station_id", "month"]).agg(
        trips_ended=("ride_id", "size"),
    ).reset_index()
    monthly_ends.append(ends)

    # each name a station was recorded under this month, with its typical location
    stations_seen = trips.groupby(["start_station_id", "start_station_name"]).agg(
        lat=("start_lat", "median"),
        lon=("start_lng", "median"),
        trips=("ride_id", "size"),
    ).reset_index()
    monthly_stations.append(stations_seen)

202401-bluebikes-tripdata.csv kept 164133 of 166699


202402-bluebikes-tripdata.csv kept 228941 of 231947


202403-bluebikes-tripdata.csv kept 258508 of 261687


202404-bluebikes-tripdata.csv kept 351586 of 356327


202405-bluebikes-tripdata.csv kept 434810 of 440534


...

202408-bluebikes-tripdata.csv kept 537661 of 538262


202409-bluebikes-tripdata.csv kept 582097 of 582679


202410-bluebikes-tripdata.csv kept 563653 of 564156


202411-bluebikes-tripdata.csv kept 397056 of 397406


202412-bluebikes-tripdata.csv kept 190399 of 190611


The cleaning log: one row per file, plus a total.

In [4]:
cleaning_log = pd.DataFrame(cleaning_log)

total = {"file": "all files"}
for column in cleaning_log.columns[1:]:
    total[column] = cleaning_log[column].sum()
cleaning_log = pd.concat([cleaning_log, pd.DataFrame([total])], ignore_index=True)

cleaning_log

,file,rows_in,no_start_station,under_1_min,over_24_hours,outside_study_year,in_previous_file,rows_out
0,202401-bluebikes-tripdata.csv,166699,141,2259,166,0,0,164133
1,202402-bluebikes-tripdata.csv,231947,294,2521,191,0,0,228941
2,202403-bluebikes-tripdata.csv,261687,38,2930,211,0,0,258508
3,202404-bluebikes-tripdata.csv,356327,65,4388,288,0,0,351586
4,202405-bluebikes-tripdata.csv,440534,34,5338,352,0,0,434810
5,202406-bluebikes-tripdata.csv,478860,40,0,500,0,123,478197
6,202407-bluebikes-tripdata.csv,542622,91,0,515,0,0,542016
7,202408-bluebikes-tripdata.csv,538262,184,0,417,0,0,537661
8,202409-bluebikes-tripdata.csv,582679,165,0,417,0,0,582097
9,202410-bluebikes-tripdata.csv,564156,178,0,325,0,0,563653


**Sanity check.** No ride should be left in two files.

In [5]:
ride_ids = pd.concat([pd.read_parquet(p, columns=["ride_id"]) for p in sorted(INTERIM_TRIPS.glob("*.parquet"))])

print(len(ride_ids), "clean trips")
assert ride_ids["ride_id"].duplicated().sum() == 0, "A ride appears in two files." 

4729057 clean trips


## 2. One row per station

Each month gave us every name a station was recorded under, and its typical (median) location that month.

* **Name:** the name used for the most trips.
* **Location:** the median of its monthly locations. A station's recorded coordinates shift slightly from trip to trip, and the median ignores the odd outlier.

In [6]:
station_names = pd.concat(monthly_stations)

# the name used for the most trips: sort the most-used first, then keep the first row per station
name_counts = station_names.groupby(["start_station_id", "start_station_name"]).agg(
    trips=("trips", "sum"),
).reset_index()
name_counts = name_counts.sort_values("trips", ascending=False)
most_used_name = name_counts.drop_duplicates(subset="start_station_id", keep="first")

# the typical location, and how many different names the station had
location = station_names.groupby("start_station_id").agg(
    lat=("lat", "median"),
    lon=("lon", "median"),
    names_used=("start_station_name", "nunique"),
).reset_index()

stations = location.merge(most_used_name[["start_station_id", "start_station_name"]], on="start_station_id")
stations = stations.rename(columns={"start_station_id": "station_id", "start_station_name": "station_name"})

print(len(stations), "stations;", (stations["names_used"] > 1).sum(), "were recorded under more than one name")

529 stations; 12 were recorded under more than one name


## 3. Trips per station per month

A file can hold a few trips that started in the previous month, so one station and month can appear in two of the monthly tables.
Adding them up with `groupby().sum()` merges those rows into one.

In [7]:
station_month = pd.concat(monthly_starts).groupby(["start_station_id", "month"]).sum().reset_index()
station_month = station_month.rename(columns={"start_station_id": "station_id"})

station_ends = pd.concat(monthly_ends).groupby(["end_station_id", "month"]).sum().reset_index()
station_ends = station_ends.rename(columns={"end_station_id": "station_id"})

station_month = station_month.merge(station_ends, on=["station_id", "month"], how="left")
station_month["trips_ended"] = station_month["trips_ended"].fillna(0).astype(int)

# when each station was in use
activity = station_month.groupby("station_id").agg(
    first_month=("month", "min"),
    last_month=("month", "max"),
    months_active=("month", "nunique"),
).reset_index()
stations = stations.merge(activity, on="station_id")

station_month.head()

,station_id,month,trips_started,member_trips,ebike_trips,weekend_trips,trips_ended
0,A32000,2024-01,301,264,103,29,318
1,A32000,2024-02,384,309,167,63,392
2,A32000,2024-03,522,355,204,163,565
3,A32000,2024-04,635,443,179,134,647
4,A32000,2024-05,1002,638,275,174,1115


## 4. Census data for each tract

The team first downloaded **ACS 1-year** tables (`2024_ACS_one_year_Boston.csv` and two `S0802` tables). They describe the whole city in **one row**: the 1-year survey is only published for areas with 65,000 people or more, so it has no census tracts.

We use the **ACS 2020–2024 5-year estimates** for every Suffolk County tract instead. The file was downloaded once from [Census Reporter](https://censusreporter.org) and is saved in `data/raw/acs/`.
Its layout is `data` → tract → table → `estimate` → column, so each value is read with a chain of keys.

| Table | Column | Meaning |
|---|---|---|
| B01003 | `B01003001` | total population |
| B19013 | `B19013001` | median household income |
| B03002 | `B03002001`, `003`, `004`, `006`, `012` | total; white, Black and Asian (not Hispanic); Hispanic or Latino |
| B08201 | `B08201001`, `B08201002` | households; households with no vehicle |

A tract the Census could not estimate income for has `None` in the file. pandas reads it as `NaN` (missing), **not 0**, so it cannot land in the lowest income group by mistake.

In [8]:
acs_json = json.load(open(RAW_ACS))
print("ACS release:", acs_json["release"]["name"], acs_json["release"]["years"])

rows = []
for geo_id, tables in acs_json["data"].items():
    rows.append({
        "GEOID": geo_id.split("US")[1],   # "14000US25025000101" becomes "25025000101", as in Lab 6
        "population": tables["B01003"]["estimate"]["B01003001"],
        "median_hh_income": tables["B19013"]["estimate"]["B19013001"],
        "race_total": tables["B03002"]["estimate"]["B03002001"],
        "white_nh": tables["B03002"]["estimate"]["B03002003"],
        "black_nh": tables["B03002"]["estimate"]["B03002004"],
        "asian_nh": tables["B03002"]["estimate"]["B03002006"],
        "hispanic": tables["B03002"]["estimate"]["B03002012"],
        "households": tables["B08201"]["estimate"]["B08201001"],
        "hh_no_vehicle": tables["B08201"]["estimate"]["B08201002"],
    })
acs = pd.DataFrame(rows)

# percentages; a tract with nobody in it gives 0 / 0, which pandas stores as NaN
acs["pct_white_nh"] = (acs["white_nh"] / acs["race_total"] * 100).round(1)
acs["pct_black_nh"] = (acs["black_nh"] / acs["race_total"] * 100).round(1)
acs["pct_asian_nh"] = (acs["asian_nh"] / acs["race_total"] * 100).round(1)
acs["pct_hispanic"] = (acs["hispanic"] / acs["race_total"] * 100).round(1)
acs["pct_hh_no_vehicle"] = (acs["hh_no_vehicle"] / acs["households"] * 100).round(1)

# the ACS reports any income of $250,000 or more as $250,001
acs["income_top_coded"] = acs["median_hh_income"] >= 250_001

print(len(acs), "tracts |", acs["median_hh_income"].isna().sum(), "with no income estimate |",
      acs["income_top_coded"].sum(), "top-coded")

ACS release: ACS 2024 5-year 2020-2024
235 tracts | 18 with no income estimate | 3 top-coded


## 5. Boston's tracts, and which tract each station is in

Suffolk County also contains Chelsea, Revere and Winthrop. A tract counts as **Boston** if its centroid is inside the City of Boston boundary.

Two Boston tracts have such irregular shapes (the Logan Airport area, and the Harbor Islands) that their centroid falls outside the city line, so they are left out. Neither has residents or a station.

Both layers are converted to a projected CRS in feet before any spatial test, as in week 7.

In [9]:
place = gpd.read_file(RAW_BOUNDARIES / "cb_2023_25_place_500k.zip")
boston = place[place["NAME"] == "Boston"].to_crs(LOCAL_CRS)
boston_shape = boston.geometry.iloc[0]

tracts = gpd.read_file(RAW_BOUNDARIES / "cb_2023_25_tract_500k.zip")
tracts = tracts[tracts["COUNTYFP"] == "025"]                  # 025 is Suffolk County
tracts = tracts[["GEOID", "NAMELSAD", "geometry"]].to_crs(LOCAL_CRS)
tracts = tracts[tracts.centroid.within(boston_shape)]         # keep the tracts in Boston

tracts = tracts.merge(acs, on="GEOID", how="left")
tracts["residential"] = tracts["population"] >= RESIDENTIAL_MIN_POP

print(len(tracts), "Boston tracts,", tracts["residential"].sum(), "of them residential")

204 Boston tracts, 190 of them residential


In [10]:
station_points = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(x=stations["lon"], y=stations["lat"]),
    crs=4326,
).to_crs(LOCAL_CRS)

station_points["in_boston"] = station_points.within(boston_shape)

# which tract each station is inside; stations outside Boston get no GEOID
station_points = gpd.sjoin(station_points, tracts[["GEOID", "geometry"]], how="left", predicate="within")
station_points = station_points.drop(columns="index_right")

in_boston = station_points[station_points["in_boston"]]
print(len(in_boston), "of", len(station_points), "stations are in Boston;",
      in_boston["GEOID"].isna().sum(), "of those are in no tract")

281 of 529 stations are in Boston; 0 of those are in no tract


## 6. Trips per tract

A tract's trips are the trips started at the stations inside it. Tracts with no station are kept, with zeros.

`trips_per_1k_residents` is left blank for non-residential tracts, where dividing by a tiny population would give a meaningless number.

In [11]:
station_month = station_month.merge(station_points[["station_id", "in_boston", "GEOID"]], on="station_id", how="left")
boston_station_month = station_month[station_month["in_boston"]]

# 2024 totals for each tract
tract_totals = boston_station_month.groupby("GEOID").agg(
    stations=("station_id", "nunique"),
    trips_started=("trips_started", "sum"),
).reset_index()

tracts = tracts.merge(tract_totals, on="GEOID", how="left")
tracts["stations"] = tracts["stations"].fillna(0).astype(int)
tracts["trips_started"] = tracts["trips_started"].fillna(0).astype(int)
tracts["trips_per_1k_residents"] = (tracts["trips_started"] / tracts["population"] * 1000).round(1)
tracts.loc[~tracts["residential"], "trips_per_1k_residents"] = None

# trips for each tract and month: one row per tract, one column per month, as in week 7
by_month = boston_station_month.groupby(["GEOID", "month"])["trips_started"].sum().unstack(fill_value=0).reset_index()
tract_trips_by_month = tracts[["GEOID"]].merge(by_month, on="GEOID", how="left").fillna(0)
month_columns = tract_trips_by_month.columns[1:]
tract_trips_by_month[month_columns] = tract_trips_by_month[month_columns].astype(int)

print(tracts["stations"].eq(0).sum(), "of", len(tracts), "Boston tracts have no station")
tracts[["GEOID", "population", "median_hh_income", "stations", "trips_started", "trips_per_1k_residents"]].head()

65 of 204 Boston tracts have no station


,GEOID,population,median_hh_income,stations,trips_started,trips_per_1k_residents
0,25025010405,6924.0,28125.0,2,42407,6124.6
1,25025110403,4969.0,132069.0,0,0,0.0
2,25025120600,2402.0,152727.0,1,11695,4868.9
3,25025060400,5695.0,207368.0,0,0,0.0
4,25025061000,2348.0,131923.0,1,6589,2806.2


## 7. Save

`data/clean/` is small enough for GitHub.

In [12]:
cleaning_log.to_csv(CLEAN / "cleaning_log.csv", index=False)
station_points.drop(columns="geometry").to_csv(CLEAN / "stations.csv", index=False)
station_month.to_csv(CLEAN / "station_month.csv", index=False)
tracts.drop(columns="geometry").to_csv(CLEAN / "boston_tracts.csv", index=False)
tracts.to_crs(4326).to_file(CLEAN / "boston_tracts.geojson", driver="GeoJSON")
tract_trips_by_month.to_csv(CLEAN / "tract_trips_by_month.csv", index=False)

for f in sorted(CLEAN.iterdir()):
    print(f.name)

boston_tracts.csv
boston_tracts.geojson
cleaning_log.csv
station_month.csv
stations.csv
tract_trips_by_month.csv
